# 02 — Short-Term Memory

Tests `memory/st_memory.py`: message buffering, history truncation, and reset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
from memory.st_memory import ShortTermMemory

## 1. Basic add & retrieve

In [ ]:
mem = ShortTermMemory(max_turns=5)

mem.add_message('user', 'Hello, I need a gift for my wife.')
mem.add_message('assistant', 'Sure! Does she have any allergies?')
mem.add_message('user', 'She is allergic to nuts.')

history = mem.get_history()
print(f'Messages in history: {len(history)}')
for msg in history:
    print(f"  [{msg['role']}]: {msg['content']}")

assert len(history) == 3
assert history[0]['role'] == 'user'
assert history[1]['role'] == 'assistant'
assert 'nuts' in history[2]['content']
print('Basic add/retrieve: PASSED')

## 2. Truncation at max_turns

In [ ]:
mem = ShortTermMemory(max_turns=3)  # keeps last 2*3=6 messages

# Add 10 pairs (20 messages total)
for i in range(10):
    mem.add_message('user', f'User message {i}')
    mem.add_message('assistant', f'Assistant reply {i}')

history = mem.get_history()
print(f'Messages stored: {len(history)} (expected <= {2*3})')

# Should only keep the last 6 messages
assert len(history) <= 6, f'Expected at most 6, got {len(history)}'

# Most recent message should be the last one added
assert 'User message 9' in history[-2]['content'] or 'reply 9' in history[-1]['content'], \
    'Most recent messages should be preserved'

print('Truncation test: PASSED')

## 3. History format (no timestamps exposed)

In [ ]:
mem = ShortTermMemory(max_turns=5)
mem.add_message('user', 'test')
history = mem.get_history()

# Each message should only have 'role' and 'content', no timestamp
for msg in history:
    assert set(msg.keys()) == {'role', 'content'}, \
        f'Expected only role+content, got: {msg.keys()}'

print('Format check (no timestamps): PASSED')

## 4. Reset

In [ ]:
mem = ShortTermMemory(max_turns=5)
mem.add_message('user', 'message before reset')
mem.add_message('assistant', 'reply before reset')

assert len(mem.get_history()) == 2

mem.reset_history()
history = mem.get_history()
assert len(history) == 0, f'After reset expected 0 messages, got {len(history)}'

print('Reset test: PASSED')

## 5. Role validation

In [ ]:
mem = ShortTermMemory(max_turns=5)

# Valid roles
for role in ['user', 'assistant', 'system']:
    mem.add_message(role, f'Message from {role}')

history = mem.get_history()
roles_seen = [m['role'] for m in history]
print('Roles in history:', roles_seen)
assert 'user' in roles_seen
assert 'assistant' in roles_seen
print('Role variety test: PASSED')

## 6. Empty history

In [ ]:
mem = ShortTermMemory()
history = mem.get_history()
assert history == [], f'New memory should have empty history, got: {history}'
print('Empty history test: PASSED')